<a href="https://colab.research.google.com/github/quyetttcoder/Fine-tune-LLM-with-small-data/blob/main/fine_tune_qwen3_8B_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q unsloth datasets trl accelerate bitsandbytes tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [ ]:
import json
import re
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from sklearn.metrics import f1_score
from tqdm import tqdm

MODEL_NAME = "unsloth/Qwen3-8B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    base_model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.7.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
from unsloth.chat_templates import get_chat_template


tokenizer = get_chat_template(
    base_tokenizer,
    chat_template = "qwen3-instruct",
)

In [ ]:
SYSTEM_PROMPT = """
You are a university admissions consultant.

Response rules:
- Only use the information provided in the Context to answer the question.
- Do not add any information that is not present in the Context.
- If the Context does not contain enough information, state clearly that there is not enough information to answer.
- Answer in Vietnamese.
- Present the response clearly, politely, and professionally, like a consultant.
"""
def formatting_prompts_func(examples):
    conversations = []

    for q, context, a in zip(
        examples["question"],
        examples["synthetic_context"],
        examples["cleaned_answer"]
    ):

        user_content = f"""
Context:
{context}

Câu hỏi:
{q}
"""

        conversations.append([
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_content.strip()
            },
            {
                "role": "assistant",
                "content": str(a)
            },
        ])

    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        for convo in conversations
    ]

    return {
        "text": texts
    }

In [ ]:
from google.colab import userdata

hf_token = userdata.get("HF_WRITE_TOKEN")

In [ ]:
from huggingface_hub import login
login(token=hf_token)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("quyetdev/QA_Admission_RAG")

README.md:   0%|          | 0.00/613 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/800k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/121k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/124k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/935 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/117 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/117 [00:00<?, ? examples/s]

In [ ]:
train_ds = dataset["train"].map(formatting_prompts_func, batched=True)
valid_ds = dataset["validation"].map(formatting_prompts_func, batched=True)
test_ds = dataset["test"].map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/935 [00:00<?, ? examples/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

In [ ]:
import os
import shutil
from transformers import TrainerCallback, EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig

LOCAL_BASE = "/content/qwen3_8B_v3"
DRIVE_BASE = "/content/drive/MyDrive/qwen3_8B_v3"

LOCAL_CKPT = os.path.join(LOCAL_BASE, "checkpoints")
LOCAL_LORA = os.path.join(LOCAL_BASE, "lora_model")

DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
DRIVE_LORA = os.path.join(DRIVE_BASE, "lora_model")

os.makedirs(LOCAL_CKPT, exist_ok=True)
os.makedirs(LOCAL_LORA, exist_ok=True)
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(DRIVE_LORA, exist_ok=True)

print("Local:", LOCAL_BASE)
print("Drive:", DRIVE_BASE)

class BackupToDriveCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):

        ckpt_name = f"checkpoint-{state.global_step}"

        src = os.path.join(args.output_dir, ckpt_name)
        dst = os.path.join(DRIVE_CKPT, ckpt_name)

        if os.path.exists(src):
            shutil.copytree(dst=dst, src=src, dirs_exist_ok=True)
            print(f"☁️ Backed up: {ckpt_name}")


trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    dataset_text_field="text",
    max_seq_length=2048,

    args=SFTConfig(
        output_dir=LOCAL_CKPT,

        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,

        eval_strategy="steps",
        eval_steps=25,

        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,

        learning_rate=2e-4,
        num_train_epochs=5,

        optim="adamw_8bit",
        lr_scheduler_type="linear",
        warmup_ratio=0.03,

        fp16=True,
        gradient_checkpointing=True,

        # Bắt buộc để load checkpoint tốt nhất
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        logging_steps=50,
        report_to="wandb",
        run_name="qwen3-8b-lora-run-3",
    ),

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.01
        )
    ]
)
trainer.add_callback(BackupToDriveCallback())

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Local: /content/qwen3_8B_v3
Drive: /content/drive/MyDrive/qwen3_8B_v3


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/935 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/117 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 935 | Num Epochs = 5 | Total steps = 585
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nguyenvanquyet18032004 (quyet_ai_engineer) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Unsloth: Restored added_tokens_decoder metadata in /content/qwen3_8B_v3/checkpoints/checkpoint-150/tokenizer_config.json.


☁️ Backed up: checkpoint-150


Unsloth: Restored added_tokens_decoder metadata in /content/qwen3_8B_v3/checkpoints/checkpoint-175/tokenizer_config.json.


☁️ Backed up: checkpoint-175


TrainOutput(global_step=175, training_loss=0.7669321550641741, metrics={'train_runtime': 2408.8859, 'train_samples_per_second': 1.941, 'train_steps_per_second': 0.243, 'total_flos': 2.291736281585664e+16, 'train_loss': 0.7669321550641741, 'epoch': 1.4957264957264957})

In [ ]:
print("💾 Saving final LoRA...")

trainer.save_model(LOCAL_LORA)
tokenizer.save_pretrained(LOCAL_LORA)

# copy final model sang Drive
shutil.copytree(
    LOCAL_LORA,
    DRIVE_LORA,
    dirs_exist_ok=True
)

print("✅ Final model saved to Drive")

In [ ]:
from google.colab import userdata

hf_token = userdata.get("HF_WRITE_TOKEN")
print("☁️ Merging LoRA and pushing to Hugging Face...")

model.push_to_hub_merged(
    "quyetdev/qwen3_8B_fine_tuned_16bit_v3",
    tokenizer,
    save_method="merged_16bit",
    token=hf_token,
)

print("✅ Final merged model pushed to Hugging Face")

☁️ Merging LoRA and pushing to Hugging Face...


config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in quyetdev/qwen3_8B_fine_tuned_16bit_v3/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   2%|1         | 23.9MB / 1.58GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [09:59<00:00, 149.75s/it]


Unsloth: Merge process complete. Saved to `/content/quyetdev/qwen3_8B_fine_tuned_16bit_v3`
✅ Final merged model pushed to Hugging Face
